# Stochastic Calculus, Step by Step · Lesson 29: Finite differences

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Replacing derivatives with differences · 3. The equation in log price · 4. The explicit scheme · 5. When it blows up · 6. The implicit scheme · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 29,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Replacing derivatives with differences

Taylor's theorem (Lesson 1) turns derivatives into differences on a grid. On $f(x) = e^x$ at $x = 0$, where every derivative is 1:

$$\frac{f(x + h) - f(x)}{h}, \quad \frac{f(x + h) - f(x - h)}{2h}, \quad \frac{f(x + h) - 2f(x) + f(x - h)}{h^2} .$$

In [ ]:
dif_rows = []
for h in (0.2, 0.1, 0.05, 0.025):
    fw = (math.exp(h) - 1) / h; ct = (math.exp(h) - math.exp(-h)) / (2 * h); sd = (math.exp(h) - 2 + math.exp(-h)) / h**2
    dif_rows.append({"h": h, "forward": fw - 1, "central": ct - 1, "second": sd - 1})
out["dif"] = {"rows": dif_rows, "fw_ratio": dif_rows[-2]["forward"] / dif_rows[-1]["forward"], "ct_ratio": dif_rows[-2]["central"] / dif_rows[-1]["central"],
              "sd_ratio": dif_rows[-2]["second"] / dif_rows[-1]["second"]}
pd.DataFrame(dif_rows)

## 3. The equation in log price

Put $y = \ln S$ and $\tau = T - t$. The Black-Scholes equation becomes one with **constant** coefficients, with $\nu = r - \tfrac12\sigma^2$:

$$V_\tau = \nu V_y + \tfrac12\sigma^2V_{yy} - rV, \qquad V(0, y) = \max(e^y - K, 0) .$$

In [ ]:
nu = r - 0.5 * sigma**2
# check on the formula, by bumps in y and tau
y0 = math.log(S0); tau0 = 0.5; hy, ht = 1e-3, 1e-4
f = lambda y, tau: float(call(math.exp(y), tau))
Vt_ = (f(y0, tau0 + ht) - f(y0, tau0 - ht)) / (2 * ht)
Vy_ = (f(y0 + hy, tau0) - f(y0 - hy, tau0)) / (2 * hy)
Vyy_ = (f(y0 + hy, tau0) - 2 * f(y0, tau0) + f(y0 - hy, tau0)) / hy**2
out["logeq"] = {"nu": nu, "Vtau": Vt_, "rhs": nu * Vy_ + 0.5 * sigma**2 * Vyy_ - r * f(y0, tau0), "Vy": Vy_, "Vyy": Vyy_}
out["logeq"]

## 4. The explicit scheme

On a grid of $M$ space steps and $N$ time steps, replace $V_\tau$ by a forward difference in $\tau$, and $V_y$, $V_{yy}$ by central differences at the old time. Each new value comes from three old ones:

$$V_i^{n+1} = \frac{p_uV_{i+1}^n + p_mV_i^n + p_dV_{i-1}^n}{1 + r\Delta\tau}, \qquad p_{u,d} = \Delta\tau\Big(\frac{\sigma^2}{2\Delta y^2} \pm \frac{\nu}{2\Delta y}\Big), \quad p_m = 1 - \frac{\sigma^2\Delta\tau}{\Delta y^2} .$$

The weights add to 1: it is a discounted average of three neighbours. It **is** a trinomial tree on a fixed grid. Taking $\Delta y = \sigma\sqrt{3\Delta\tau}$ gives Lesson 23's probabilities. As with the tree, the error is untidy until the strike sits on a grid point; then it quarters each time $\Delta y$ halves.

In [ ]:
def grid(M_):
    half = 5 * sigma * math.sqrt(T)
    y = math.log(S0) + np.linspace(-half, half, M_ + 1)
    return y, y[1] - y[0]


def interp_at_S0(y, v):
    return float(np.interp(math.log(S0), y, v))


def explicit(M_, N_, kind="call", american=False, sig=sigma, snap=False):
    # March the log-price equation forward in tau, on a fixed grid centred at S0.
    # snap=True nudges the spacing so that the strike is exactly on a grid point.
    nu_ = r - 0.5 * sig**2
    half = 5 * sig * math.sqrt(T)
    dy = 2 * half / M_
    if snap:
        gap = math.log(K / S0)
        dy = gap / max(1, round(gap / dy))
    y = math.log(S0) + dy * (np.arange(M_ + 1) - M_ // 2)
    dtau = T / N_
    a_ = dtau * sig**2 / (2 * dy**2)
    b_ = dtau * nu_ / (2 * dy)
    pu_, pm_, pd_ = a_ + b_, 1 - 2 * a_, a_ - b_
    s_ = np.exp(y)
    pay = np.maximum(s_ - K, 0) if kind == "call" else np.maximum(K - s_, 0)
    v = pay.copy()
    for n in range(1, N_ + 1):
        new = v.copy()
        new[1:-1] = (pu_ * v[2:] + pm_ * v[1:-1] + pd_ * v[:-2]) / (1 + r * dtau)
        tau = n * dtau
        if kind == "call":
            new[0], new[-1] = 0.0, s_[-1] - K * math.exp(-r * tau)
        else:
            new[0], new[-1] = K * math.exp(-r * tau) - s_[0], 0.0
            if american:
                new[0] = K - s_[0]
        if american:
            new = np.maximum(new, pay)
        v = new
        if not np.isfinite(v).all() or np.abs(v).max() > 1e12:
            return float("nan"), sig**2 * dtau / dy**2
    return interp_at_S0(y, v), sig**2 * dtau / dy**2


bs_c, bs_p = float(call(S0, T)), float(put(S0, T))
ex_rows = []
for M_, N_ in ((100, 1000), (200, 4000), (400, 16000), (800, 64000)):
    c_, cfl = explicit(M_, N_)
    c2, _ = explicit(M_, N_, snap=True)
    ex_rows.append({"M": M_, "N": N_, "ratio": cfl, "call": c_, "error": c_ - bs_c, "snap": c2 - bs_c})
out["explicit"] = {"rows": ex_rows, "bs": bs_c, "err_ratio": ex_rows[-2]["snap"] / ex_rows[-1]["snap"]}
# the same weights as Lesson 23's tree when dy = sigma sqrt(3 dtau)
dt3 = 0.01; dy3 = sigma * math.sqrt(3 * dt3)
out["explicit"]["tree"] = {"pu": dt3 * (sigma**2 / (2 * dy3**2) + nu / (2 * dy3)), "pm": 1 - sigma**2 * dt3 / dy3**2, "pd": dt3 * (sigma**2 / (2 * dy3**2) - nu / (2 * dy3))}
pd.DataFrame(ex_rows)

## 5. When it blows up

The scheme is an average only while $p_m \geq 0$, that is, while $\sigma^2\Delta\tau/\Delta y^2 \leq 1$. Push past that, and the round-off error doubles at every step. Below, $M = 200$ space steps, and fewer and fewer time steps. The limit is at $N = \sigma^2TM^2/(10\sigma\sqrt T)^2 = M^2/100 = 400$.

In [ ]:
st_rows = []
for N_ in (2000, 800, 450, 400, 395, 380, 300):
    c_, cfl = explicit(200, N_)
    ok = c_ == c_ and abs(c_ - bs_c) < 0.1
    st_rows.append({"N": N_, "ratio": cfl, "pm": round(1 - cfl, 10) + 0.0, "amp": round(abs(1 - 2 * cfl), 10), "call": f"{c_:.4f}" if ok else "blown up"})


def explicit_profile(M_, N_, steps):
    half = 5 * sigma * math.sqrt(T)
    y = math.log(S0) + np.linspace(-half, half, M_ + 1)
    dy = y[1] - y[0]; dtau = T / N_
    a_ = dtau * sigma**2 / (2 * dy**2); b_ = dtau * nu / (2 * dy)
    pu_, pm_, pd_ = a_ + b_, 1 - 2 * a_, a_ - b_
    v = np.maximum(np.exp(y) - K, 0)
    for n in range(1, steps + 1):
        new = v.copy()
        new[1:-1] = (pu_ * v[2:] + pm_ * v[1:-1] + pd_ * v[:-2]) / (1 + r * dtau)
        new[0], new[-1] = 0.0, math.exp(y[-1]) - K * math.exp(-r * n * dtau)
        v = new
    return np.exp(y), v, a_ * 2


s_ok, v_ok, cfl_ok = explicit_profile(200, 800, 800)
s_bad, v_bad, cfl_bad = explicit_profile(200, 340, 90)
mask = (s_ok > 60) & (s_ok < 170)
out["stab"] = {"rows": st_rows, "fig": {"S": s_ok[mask].tolist(), "ok": v_ok[mask].tolist(), "bad": np.clip(v_bad[mask], -40, 90).tolist()}, "cfl_bad": cfl_bad, "cfl_ok": cfl_ok, "limit_N": 400,
               "bad_max": float(np.abs(v_bad).max())}
pd.DataFrame(st_rows)

## 6. The implicit scheme

Use the differences at the **new** time instead. Each step now needs a system of equations, a tridiagonal one, that takes only about $3M$ operations to solve. In exchange, there is no blow-up, however big the time step. The Crank-Nicolson scheme averages old and new, and gains an order of accuracy in time.

In [ ]:
def thomas(lo, di, up, rhs):
    # Solve a tridiagonal system with constant bands.
    n = rhs.size
    c_, d_ = np.empty(n), np.empty(n)
    c_[0] = up / di; d_[0] = rhs[0] / di
    for i in range(1, n):
        m = di - lo * c_[i - 1]
        c_[i] = up / m
        d_[i] = (rhs[i] - lo * d_[i - 1]) / m
    x = np.empty(n)
    x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    return x


def theta_scheme(M_, N_, w):
    # w = 0 explicit, w = 1 implicit, w = 1/2 Crank-Nicolson.
    half = 5 * sigma * math.sqrt(T)
    y = math.log(S0) + np.linspace(-half, half, M_ + 1)
    dy = y[1] - y[0]; dtau = T / N_
    a_ = sigma**2 / (2 * dy**2); b_ = nu / (2 * dy)
    up, md, lo = a_ + b_, -2 * a_ - r, a_ - b_
    s_ = np.exp(y)
    v = np.maximum(s_ - K, 0)
    for n in range(1, N_ + 1):
        hi_new = s_[-1] - K * math.exp(-r * n * dtau); hi_old = s_[-1] - K * math.exp(-r * (n - 1) * dtau)
        rhs = v[1:-1] + (1 - w) * dtau * (up * v[2:] + md * v[1:-1] + lo * v[:-2])
        rhs[-1] += w * dtau * up * hi_new
        inner = thomas(-w * dtau * lo, 1 - w * dtau * md, -w * dtau * up, rhs)
        v = np.concatenate([[0.0], inner, [hi_new]])
    return interp_at_S0(y, v), sigma**2 * dtau / dy**2


imp_rows = []
for M_, N_ in ((200, 50), (200, 100), (200, 200), (200, 400), (200, 1000)):
    i_, ratio_ = theta_scheme(M_, N_, 1.0)
    cn_, _ = theta_scheme(M_, N_, 0.5)
    e_ = explicit(M_, N_)[0]
    imp_rows.append({"N": N_, "ratio": ratio_, "implicit": i_ - bs_c, "cn": cn_ - bs_c, "explicit": f"{e_ - bs_c:.5f}" if e_ == e_ and abs(e_ - bs_c) < 1 else "blown up"})
out["implicit"] = {"rows": imp_rows}
# American put on the explicit grid
am = explicit(400, 16000, "put", american=True)[0]
eu = explicit(400, 16000, "put")[0]
out["american"] = {"fd": am, "european": eu, "bs_put": bs_p}
pd.DataFrame(imp_rows)

In [ ]:
# Chart data for the video (no random numbers)
out["stab"]["fig"]["ticks"] = [{"x": float(x_), "label": f"{x_:.0f}"} for x_ in (60, 80, 100, 120, 140, 160)]

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Use Taylor's theorem to show that the central difference $\frac{f(x + h) - f(x - h)}{2h}$ has error $\tfrac16f'''(x)h^2 + \dots$. Compare with the forward difference's error. Check the ratio with $f = e^x$, $h = 0.1$.

**Exercise 2.** Do the same for the second difference: show that $\frac{f(x + h) - 2f(x) + f(x - h)}{h^2} = f'' + \tfrac1{12}f''''h^2 + \dots$.

**Exercise 3.** Starting from $V_t + rSV_S + \tfrac12\sigma^2S^2V_{SS} = rV$, change variables to $y = \ln S$ and $\tau = T - t$. Derive the log equation. Then discretise it to get the explicit scheme, including the factor $1/(1 + r\Delta\tau)$. Show that the three weights add to 1.

**Exercise 4.** **Stability.** Try a wave $V_i^n = A^ne^{\mathrm ii\xi\Delta y}$ in the scheme with $\nu = r = 0$, where $\mathrm i^2 = -1$. Show $A = 1 - 2c\sin^2(\xi\Delta y/2)$, with $c = \sigma^2\Delta\tau/\Delta y^2$. For which $c$ is $|A| \leq 1$ at every wave? Explain the result by how an average of neighbours behaves.

**Exercise 5.** Halving $\Delta y$ forces $\Delta\tau$ down by 4 to stay stable. If the space error is $O(\Delta y^2)$, how does the total work grow as the error falls? Compare with a Monte Carlo error of $O(\sqrt{1/\text{work}})$: which one wins, and at what error, for one share?

**Exercise 6.** Why do we need boundary values at the grid's two ends? Justify $V = 0$ at the bottom and $V = S - Ke^{-r\tau}$ at the top for a call. Write the corresponding values for a put. How far out must the grid go for the choice not to matter? (Compare with the standard deviation of $\ln S_T$.)

**Exercise 7.** (Code) Use the explicit scheme to price the European put, then add an early-exercise check for the American put, on grids $(M, N)$ of $(100, 1000)$, $(200, 4000)$, $(400, 16000)$. Compare with the formula and with Lesson 22's tree value 9.478. What is the error's order?

**Exercise 8.** (Code) With $M = 200$, compare the explicit, implicit and Crank-Nicolson schemes at $N = 50, 100, 200, 400, 1000$. Which give sensible answers when the explicit limit is broken? Estimate the time-order of the implicit and Crank-Nicolson errors, keeping $M = 800$ fixed.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")